<a href="https://colab.research.google.com/github/ahlqui/VeloxChemColabs/blob/main/AtomicCharges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Calculate partial charges


This notebook requires that you have a molecular structure. Two options:

Define Molecule as XYZ-coordinates: Can be generatred with free software such as www.avogadro.cc

Define Molecule with SMILES code, A molecule can be defined using a SMILES code (example below). We have two suggested ways to generate smiles from structure. 1) Sketch your molecule at https://www.rcsb.org/chemical-sketch and the SMILES code will be shown right below the structure. 2) Build your molecule at https://molview.org/ , go to Tools/Information Card and it will show you the SMILES code. Just copy/paste it into the SMILES box below.

Example of xyz-coordinates for CO2:

O 0.00 0.00 0.00

C 0.00 0.00 1.20

O 0.00 0.00 2.40

Example of SMILES for CO2:

O=C=O

In [1]:
# Local environment check for QuimicaAutomocio P2.
# The original Colab version used a Colab-only installation cell.
# In local Jupyter, skip that installation cell and use the conda environment instead:
#   conda activate quimica-echem

import sys
try:
    import veloxchem as vlx
    print(f"VeloxChem imported correctly: {getattr(vlx, '__version__', 'unknown')}")
    print(f"Python executable: {sys.executable}")
except Exception as exc:
    raise RuntimeError(
        "VeloxChem is not available in this kernel. Activate the conda environment "
        "with `conda activate quimica-echem`, start Jupyter from that terminal, "
        "and select the quimica-echem kernel."
    ) from exc

VeloxChem imported correctly: 1.0rc4
Python executable: c:\Users\Nayanika\miniconda3\envs\quimica-echem\python.exe


In [2]:
xyz_coordinates = """
N              1.145826434240         0.076040617895        -0.866510214564
C              0.542180013676        -0.253553970369         0.328076125026
C             -0.812180803268         0.183435680488         0.248804170816
C             -0.975924575717         0.783647103455        -1.046774546644
C              0.233197095678         0.701702839407        -1.699933897691
C              0.627086617292         1.165821334082        -3.062316921353
C             -1.659509446002        -0.033463443742         1.350306628380
C             -1.152303769706        -0.666345037106         2.480975385544
C              0.192029976618        -1.090810482875         2.537203649871
C              1.055965876964        -0.890411345947         1.463229371191
H              2.109680253919        -0.114560157930        -1.103988190464
H             -1.882552111687         1.226929857348        -1.453438685761
H             -0.229300080653         1.631471908750        -3.568415733552
H              1.441135976413         1.910712871040        -3.020599362874
H              0.979021039689         0.330399029672        -3.692763186679
H             -2.702462402193         0.291691715994         1.316192928924
H             -1.802760846032        -0.839428928435         3.341831144390
H              0.562156000137        -1.585287734052         3.438646391744
H              2.097266855107        -1.219390148201         1.507387413976
"""


In [3]:
import veloxchem as vlx

molecule = vlx.Molecule.read_str(xyz_coordinates)
print('Structure of the molecule entered: ')
molecule.show(atom_indices=True)

Structure of the molecule entered: 


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
#@title DFT single point and charge calculation
#@markdown For classroom tests, start with a small basis such as STO-3G or DEF2-SVP.


#molecule.set_charge(1)

basis_set = 'DEF2-SVP' #@param  ['STO-3G', 'DEF2-SV(P)', 'DEF2-SVP', 'DEF2-SVPD', 'DEF2-TZVP']
basis = vlx.MolecularBasis.read(molecule, basis_set)
scf_drv = vlx.ScfRestrictedDriver()
method_dict = {'conv_thresh': 1e-4}
scf_drv.update_settings(method_dict)
mute_output = True # @param {type:"boolean"}
if mute_output:
    scf_drv.ostream.mute()
functional = 'B3LYP' #@param ['SLATER', 'BLYP', 'B3LYP', 'PBE', 'PBE0', 'BP86']
scf_drv.xcfun = functional
scf_results = scf_drv.compute(molecule, basis)

# VeloxChem 1.0 separates ESP and RESP drivers. Older Colab notebooks used
# RespChargesDriver(..., "esp"), which is now deprecated and raises an error.
esp_drv = vlx.EspChargesDriver()
esp_drv.ostream.mute()
esp_charges = esp_drv.compute(molecule, basis, scf_results)

resp_drv = vlx.RespChargesDriver()
resp_drv.ostream.mute()
resp_charges = resp_drv.compute(molecule, basis, scf_results)

chelpg_drv = vlx.EspChargesDriver()
chelpg_drv.ostream.mute()
chelpg_drv.grid_type = "chelpg"
chelpg_drv.equal_charges = "1=1, 1=1"
chelpg_charges = chelpg_drv.compute(molecule, basis, scf_results)

for title, charges in [
    ("ESP charge", esp_charges),
    ("RESP charge", resp_charges),
    ("CHELPG charge", chelpg_charges),
]:
    print(f"Atom      {title}")
    print(20 * "-")
    for label, charge in zip(molecule.get_labels(), charges):
        print(f"{label :s} {charge : 18.6f}")
    print(20 * "-")
    print(f"Total: {charges.sum() : 13.6f}\n")

Atom      ESP charge
--------------------
N          -0.427797
C           0.141361
C           0.238873
C          -0.554708
C           0.330996
C          -0.313139
C          -0.250140
C          -0.151549
C          -0.109758
C          -0.255486
H           0.341845
H           0.207414
H           0.107175
H           0.089708
H           0.088571
H           0.147177
H           0.117041
H           0.111243
H           0.141172
--------------------
Total:      0.000000

Atom      RESP charge
--------------------
N          -0.199331
C           0.045228
C           0.093428
C          -0.335156
C           0.067389
C          -0.103997
C          -0.158110
C          -0.160629
C          -0.103076
C          -0.199882
H           0.280258
H           0.159676
H           0.051217
H           0.051217
H           0.051217
H           0.120997
H           0.112285
H           0.105580
H           0.121688
--------------------
Total:      0.000000

Atom      CHELPG charge
-------

In [5]:
#@title Visualize the charges on molecule
import py3Dmol
viewer = py3Dmol.view(linked=False, width=500, height=500)

charge_type = 'ESP' # @param ["RESP", "ESP", "CHELPG"]
charge_map = {
    'ESP': esp_charges,
    'RESP': resp_charges,
    'CHELPG': chelpg_charges,
}
charges = charge_map[charge_type]
print(f'{charge_type} charges:')
viewer.addModel(molecule.get_xyz_string(), "xyz")
for i, charge in enumerate(charges):
    viewer.addLabel(
        "{:.2f}".format(charge),
        {'backgroundOpacity': 0, 'fontColor': 'black', 'backgroundColor': 'white'},
        {'index': i},
    )
viewer.setStyle({"stick": {}, "sphere": {"radius": 0.4}})
viewer.rotate(-45, "y")
viewer.zoomTo()
viewer.show()

ESP charges:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.